# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample

plt.style.use('seaborn-v0_8-darkgrid')

def locate_project_paths() -> Tuple[Path, Path]:
    """Discover project root and processed data directory by walking up from CWD."""
    start = Path.cwd()
    search_roots = [start, *start.parents]
    for root in search_roots:
        candidate = root / 'data' / 'processed'
        if candidate.exists():
            return root, candidate
    raise FileNotFoundError('Unable to locate data/processed directory from current path')

PROJECT_ROOT, PROCESSED_DIR = locate_project_paths()
IMAGES_DIR = PROJECT_ROOT / 'images'
RESULTS_DIR = PROJECT_ROOT / 'results'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'VO2peak_abs'
TIME_WINDOW = (300.0, 1000.0)
WR_LEVELS = (30, 80)
MIN_SAMPLES_PER_LEVEL = 20
MIN_VALID_FRACTION = 0.6
MIN_SUBJECTS_PER_GROUP = 25

TIME_COLUMN_CANDIDATES = ('Time', 'time', 't')
WORKRATE_COLUMN_CANDIDATES = ('WorkRate', 'WR', 'work_rate')
EXCLUDED_COLUMNS = {'Time', 'time', 't', 'WorkRate', 'workrate', 'WR', 'wr', 'work_rate'}

In [ ]:
GAS_EXCHANGE_VENTILATION = [
    'VO2pulm', 'VO2_movMean', 'VCO2', 'FO2et', 'FCO2et', 'Vent', 'VT', 'BF'
]
HR_ONLY = ['HR']
HEMODYNAMICS = [
    'SV', 'CO', 'dPdt', 'SPTI', 'RPP', 'DPTI', 'DPTISPTI', 'LVET', 'Cwk',
    'Rp', 'TPR', 'fiSYS', 'fiMAP', 'fiDIA', 'sBP', 'mBP', 'dBP'
]
NIRS = ['absO2Hb_Rx1', 'absHHb_Rx1', 'TSI_Rx1_Tx1Tx2Tx3', 'TSI_FF_Rx1_Tx1Tx2Tx3']
ALL_SIGNALS = sorted({
    *HR_ONLY, *GAS_EXCHANGE_VENTILATION, *HEMODYNAMICS, *NIRS
})
NO_GAS_EXCHANGE = sorted(set(ALL_SIGNALS) - set(GAS_EXCHANGE_VENTILATION))
HR_PLUS_NIRS = sorted(set(HR_ONLY + NIRS))
HEMO_PLUS_NIRS = sorted(set(HEMODYNAMICS + NIRS))
HR_PLUS_BF = sorted(set(HR_ONLY + ['BF']))

SENSOR_GROUPS: Dict[str, Sequence[str]] = {
    'all_signals': ALL_SIGNALS,
    'gas_exchange_ventilation': GAS_EXCHANGE_VENTILATION,
    'hr_only': HR_ONLY,
    'hemodynamics': HEMODYNAMICS,
    'nirs': NIRS,
    'no_gas_exchange': NO_GAS_EXCHANGE,
    'hr_plus_nirs': HR_PLUS_NIRS,
    'hemo_plus_nirs': HEMO_PLUS_NIRS,
    'hr_plus_bf': HR_PLUS_BF,
}

In [ ]:
def detect_column(columns: Iterable[str], candidates: Sequence[str]) -> str:
    """Return the first matching column from candidates; raise if none found."""
    lower_map = {col.lower(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'None of the candidate columns {candidates} located in dataframe')

def list_subject_files(processed_dir: Path) -> List[str]:
    """List subject identifiers (e.g., CT_01) that have both data and target files."""
    subjects = []
    for data_path in sorted(processed_dir.glob('CT_*_data.csv')):
        subject_id = data_path.stem.rsplit('_', 1)[0]
        target_path = processed_dir / f'{subject_id}_targets.csv'
        if target_path.exists():
            subjects.append(subject_id)
    if not subjects:
        raise FileNotFoundError('No subject data/targets pairs discovered in processed directory')
    return subjects

def load_subject(subject_id: str) -> Tuple[pd.DataFrame, float]:
    """Load a subject's time-series dataframe and VO2peak_abs target value."""
    data_path = PROCESSED_DIR / f'{subject_id}_data.csv'
    target_path = PROCESSED_DIR / f'{subject_id}_targets.csv'
    if not data_path.exists() or not target_path.exists():
        raise FileNotFoundError(f'Missing data or target file for {subject_id}')
    data_df = pd.read_csv(data_path)
    target_df = pd.read_csv(target_path)
    if TARGET_COLUMN not in target_df.columns:
        raise KeyError(f'Target column {TARGET_COLUMN} absent in {target_path.name}')
    target_value = float(target_df[TARGET_COLUMN].iloc[0])
    return data_df, target_value

In [ ]:
def slice_prbs_window(df: pd.DataFrame, time_col: str, window: Tuple[float, float]) -> pd.DataFrame:
    """Slice the PRBS-only time window (300–1000 s) to isolate controlled perturbations."""
    start, stop = window
    window_df = df[(df[time_col] >= start) & (df[time_col] <= stop)].copy()
    if window_df.empty:
        raise ValueError(f'No samples found within time window {window}')
    return window_df

def select_numeric_signals(df: pd.DataFrame, group_columns: Sequence[str]) -> List[str]:
    """Filter the requested signal columns to those present and numeric in the dataframe."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    available = [col for col in group_columns if col in numeric_cols]
    return available

def extract_wr_conditioned_features(
    df: pd.DataFrame,
    signal_columns: Sequence[str],
    wr_column: str,
    min_samples: int,
) -> Dict[str, float]:
    """Compute WR-conditioned statistics per signal; insufficient samples yield NaN placeholders."""
    features: Dict[str, float] = {}
    lower_wr, upper_wr = sorted(WR_LEVELS)
    for signal in signal_columns:
        mean_lower = np.nan
        mean_upper = np.nan
        for wr in WR_LEVELS:
            wr_values = df.loc[df[wr_column] == wr, signal].dropna()
            mean_name = f'{signal}_mean_wr{wr}'
            std_name = f'{signal}_std_wr{wr}'
            features[mean_name] = np.nan
            features[std_name] = np.nan
            if wr_values.shape[0] >= min_samples:
                features[mean_name] = float(wr_values.mean())
                features[std_name] = float(wr_values.std(ddof=1))
                if wr == lower_wr:
                    mean_lower = features[mean_name]
                else:
                    mean_upper = features[mean_name]
        delta_name = f'{signal}_mean_delta_wr{upper_wr}_minus_wr{lower_wr}'
        features[delta_name] = (mean_upper - mean_lower) if not np.isnan(mean_lower) and not np.isnan(mean_upper) else np.nan
    return features

In [ ]:
def build_dataset(subject_ids: Sequence[str], group_columns: Sequence[str]) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """Assemble subject-level feature matrix and target vector for a sensor group."""
    feature_rows: List[Dict[str, float]] = []
    targets: Dict[str, float] = {}
    retained_columns: List[str] = []
    skipped_subjects: List[str] = []
    for subject_id in subject_ids:
        data_df, target_value = load_subject(subject_id)
        time_col = detect_column(data_df.columns, TIME_COLUMN_CANDIDATES)
        wr_col = detect_column(data_df.columns, WORKRATE_COLUMN_CANDIDATES)
        prbs_df = slice_prbs_window(data_df, time_col, TIME_WINDOW)
        if group_columns == ALL_SIGNALS:
            candidate_signals = [col for col in prbs_df.columns if col not in EXCLUDED_COLUMNS]
            signal_columns = select_numeric_signals(prbs_df, candidate_signals)
        else:
            signal_columns = select_numeric_signals(prbs_df, group_columns)
        if not signal_columns:
            skipped_subjects.append(subject_id)
            continue
        features = extract_wr_conditioned_features(prbs_df, signal_columns, wr_col, MIN_SAMPLES_PER_LEVEL)
        features['subject_id'] = subject_id
        feature_rows.append(features)
        targets[subject_id] = target_value
    if not feature_rows:
        raise ValueError('No subjects with valid signals for this group')
    feature_df = pd.DataFrame(feature_rows).set_index('subject_id')
    non_null_fraction = feature_df.notna().mean()
    keep_cols = non_null_fraction[non_null_fraction >= MIN_VALID_FRACTION].index.tolist()
    if not keep_cols:
        raise ValueError('Feature set empty after removing columns with excessive missingness')
    feature_df = feature_df[keep_cols]
    retained_columns = keep_cols
    target_series = pd.Series(targets, name=TARGET_COLUMN).loc[feature_df.index]
    if skipped_subjects:
        print(f"For the sensor group {group_columns}: Skipped {len(skipped_subjects)} subjects without required signals: {', '.join(skipped_subjects)}")
    return feature_df, target_series, retained_columns

In [ ]:
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    """Compute regression summary metrics."""
    residuals = predicted - actual
    mse = mean_squared_error(actual, predicted)
    rmse = float(np.sqrt(mse))
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': rmse,
        'R2': r2_score(actual, predicted),
        'MeanResidual': float(residuals.mean()),
        'StdResidual': float(residuals.std(ddof=1)),
    }

class FoldwiseUnitRescaler(BaseEstimator, TransformerMixin):
    """Divide each feature by train-fold standard deviation (unit-aware fold-wise rescaling)."""

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).astype(float)
        scales = X_df.std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
        self.scales_ = scales.to_numpy(dtype=float)
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return X_arr / self.scales_

def build_pipeline(model_name: str) -> Pipeline:
    """Build leakage-safe preprocessing + estimator pipeline."""
    steps = [
        ('unit_rescaler', FoldwiseUnitRescaler()),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
    if model_name == 'Ridge':
        steps.append(('model', Ridge()))
    elif model_name == 'PLS':
        steps.append(('model', PLSRegression()))
    elif model_name == 'ElasticNet':
        steps.append(('model', ElasticNet(max_iter=50000, tol=1e-4, random_state=42)))
    else:
        raise ValueError(f'Unknown model name: {model_name}')
    return Pipeline(steps)

def build_param_grid(model_name: str, X_train: pd.DataFrame) -> Dict[str, Sequence]:
    """Return model-specific tuning grid for inner CV."""
    if model_name == 'Ridge':
        return {'model__alpha': np.logspace(-3, 2, 12)}
    if model_name == 'PLS':
        max_components = max(1, min(10, X_train.shape[1], X_train.shape[0]))
        return {'model__n_components': list(range(1, max_components + 1))}
    if model_name == 'ElasticNet':
        return {
            'model__alpha': np.logspace(-3, 1, 12),
            'model__l1_ratio': [0.2, 0.4, 0.6, 0.8],
        }
    raise ValueError(f'Unknown model name: {model_name}')

def nested_loso_tune_and_predict(
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, Dict[str, float], pd.DataFrame]:
    """Run nested LOSO (outer) with inner 5-fold tuning and return predictions/metrics/params."""
    loo = LeaveOneOut()
    prediction_records: List[Dict[str, float]] = []
    param_records: List[Dict[str, float]] = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline = build_pipeline(model_name)
        param_grid = build_param_grid(model_name, X_train)
        n_splits = min(5, len(X_train))

        if n_splits >= 2 and all(len(v) > 0 for v in param_grid.values()):
            inner_cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=inner_cv,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                refit=True,
            )
            search.fit(X_train, y_train)
            best_estimator = search.best_estimator_
            best_params = search.best_params_
        else:
            best_estimator = pipeline.fit(X_train, y_train)
            best_params = {
                key: values[0] if isinstance(values, (list, tuple, np.ndarray)) else values
                for key, values in param_grid.items()
            }

        y_pred = float(best_estimator.predict(X_test)[0])
        subject_id = X.index[test_idx][0]
        prediction_records.append({
            'subject_id': subject_id,
            'actual_vo2peak': float(y_test.iloc[0]),
            'predicted_vo2peak': y_pred,
        })

        cleaned_params = {param.replace('model__', ''): value for param, value in best_params.items()}
        cleaned_params['subject_id'] = subject_id
        param_records.append(cleaned_params)

    prediction_df = pd.DataFrame(prediction_records).set_index('subject_id').sort_index()
    prediction_df['residual'] = prediction_df['predicted_vo2peak'] - prediction_df['actual_vo2peak']
    prediction_df['mean_vo2'] = (prediction_df['predicted_vo2peak'] + prediction_df['actual_vo2peak']) / 2.0
    metrics = compute_metrics(prediction_df['actual_vo2peak'].values, prediction_df['predicted_vo2peak'].values)
    params_df = pd.DataFrame(param_records).set_index('subject_id').sort_index()
    return prediction_df, metrics, params_df

In [ ]:
def plot_pred_vs_actual(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    """Scatter plot highlighting model agreement with a 1:1 reference line."""
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, color='navy', alpha=0.75)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, linestyle='--', color='gray', label='Ideal 1:1')
    plt.xlabel('Actual VO2peak_abs (L/min)')
    plt.ylabel('Predicted VO2peak_abs (L/min)')
    plt.title(title)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_bland_altman(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    """Bland–Altman plot visualizing bias and agreement limits between predictions and targets."""
    mean_values = (y_true + y_pred) / 2.0
    residuals = y_pred - y_true
    mean_residual = residuals.mean()
    std_residual = residuals.std(ddof=1)
    loa = 1.96 * std_residual
    plt.figure(figsize=(6, 4))
    plt.scatter(mean_values, residuals, color='darkorange', alpha=0.75)
    plt.axhline(mean_residual, color='black', linestyle='--', linewidth=1, label='Mean bias')
    plt.axhline(mean_residual + loa, color='red', linestyle='--', linewidth=0.8, label='Upper LOA')
    plt.axhline(mean_residual - loa, color='red', linestyle='--', linewidth=0.8, label='Lower LOA')
    plt.xlabel('Mean VO2peak_abs (L/min)')
    plt.ylabel('Residual (Predicted - Actual)')
    plt.title(title)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_residual_hist(residuals: pd.Series, title: str, save_path: Path) -> None:
    """Histogram summarizing residual distribution to spotlight systematic error."""
    plt.figure(figsize=(6, 4))
    bins = max(5, min(30, residuals.count()))
    plt.hist(residuals, bins=bins, color='slateblue', alpha=0.8, edgecolor='white')
    plt.axvline(residuals.mean(), color='black', linestyle='--', linewidth=1, label='Mean residual')
    plt.xlabel('Residual (Predicted - Actual)')
    plt.ylabel('Count')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [ ]:
def safe_name(value: str) -> str:
    """Return a filesystem-friendly token for filenames."""
    return value.replace(' ', '_').replace('/', '_')

def run_ablation(groups: Dict[str, Sequence[str]], models: Sequence[str]) -> pd.DataFrame:
    """Execute ablation with leakage-safe nested LOSO + inner tuning."""
    subject_ids = list_subject_files(PROCESSED_DIR)
    summary_records: List[Dict[str, object]] = []
    B = 1000
    for group_name, columns in groups.items():
        try:
            X_group, y_group, retained_features = build_dataset(subject_ids, columns)
        except ValueError as exc:
            print(f'Skipping group {group_name} due to: {exc}')
            continue

        if len(y_group) < MIN_SUBJECTS_PER_GROUP:
            print(
                f"Skipping group {group_name}: only {len(y_group)} subjects with required signals "
                f"(< {MIN_SUBJECTS_PER_GROUP})."
            )
            continue

        for model_name in models:
            preds_df, metrics, params_df = nested_loso_tune_and_predict(X_group, y_group, model_name)
            file_token = f"ablation_{safe_name(group_name)}_{safe_name(model_name)}"

            preds_path = RESULTS_DIR / f'{file_token}_loso_predictions.csv'
            params_path = RESULTS_DIR / f'{file_token}_selected_params.csv'
            preds_df.to_csv(preds_path)
            params_df.to_csv(params_path)
            plot_pred_vs_actual(
                preds_df['actual_vo2peak'],
                preds_df['predicted_vo2peak'],
                title=f'{group_name} × {model_name}: Predictions vs Actual',
                save_path=IMAGES_DIR / f'{file_token}_pred_vs_actual.png',
            )
            plot_bland_altman(
                preds_df['actual_vo2peak'],
                preds_df['predicted_vo2peak'],
                title=f'{group_name} × {model_name}: Bland-Altman',
                save_path=IMAGES_DIR / f'{file_token}_bland_altman.png',
            )
            plot_residual_hist(
                preds_df['residual'],
                title=f'{group_name} × {model_name}: Residual Histogram',
                save_path=IMAGES_DIR / f'{file_token}_residual_hist.png',
            )

            actual = preds_df['actual_vo2peak'].to_numpy(dtype=float)
            predicted = preds_df['predicted_vo2peak'].to_numpy(dtype=float)
            n_samples = len(preds_df)
            bootstrap_mae = np.empty(B)
            bootstrap_rmse = np.empty(B)
            bootstrap_r2 = np.empty(B)
            pairs = np.column_stack([actual, predicted])
            for i in range(B):
                sample = resample(pairs, replace=True, n_samples=n_samples)
                sample_actual = sample[:, 0]
                sample_predicted = sample[:, 1]
                bootstrap_mae[i] = mean_absolute_error(sample_actual, sample_predicted)
                bootstrap_rmse[i] = float(np.sqrt(mean_squared_error(sample_actual, sample_predicted)))
                bootstrap_r2[i] = r2_score(sample_actual, sample_predicted)
            mae_ci = tuple(np.percentile(bootstrap_mae, [2.5, 97.5]))
            rmse_ci = tuple(np.percentile(bootstrap_rmse, [2.5, 97.5]))
            r2_ci = tuple(np.percentile(bootstrap_r2, [2.5, 97.5]))

            summary_records.append({
                'Group': group_name,
                'Model': model_name,
                'NSubjects': len(preds_df),
                'NFeatures': len(retained_features),
                **metrics,
                'mae_ci': mae_ci,
                'rmse_ci': rmse_ci,
                'r2_ci': r2_ci,
            })
    if not summary_records:
        raise RuntimeError('Ablation study produced no results; check group definitions or data availability.')
    summary_df = pd.DataFrame(summary_records).sort_values(['Group', 'Model']).reset_index(drop=True)
    summary_path = RESULTS_DIR / 'ablation_summary_metrics.csv'
    summary_df.to_csv(summary_path, index=False)
    return summary_df

In [ ]:
MODELS = ['Ridge', 'ElasticNet', 'PLS']
ablation_summary = run_ablation(SENSOR_GROUPS, MODELS)

best_model_summary = (
    ablation_summary
    .sort_values(['Group', 'RMSE', 'MAE', 'R2'], ascending=[True, True, True, False])
    .drop_duplicates(subset=['Group'], keep='first')
    .reset_index(drop=True)
)

best_model_summary

In [ ]:
# save this summary table into csv file to be used in the paper
best_model_summary_path = RESULTS_DIR / 'ablation_best_models_summary.csv'
best_model_summary.to_csv(best_model_summary_path, index=False)

